# 03 — Técnicas de Melhoria Visual de Espectrogramas STFT

Avalia quatro técnicas sobre os espectrogramas (`Equalização de Histograma`, `CLAHE`, `Brilho` e `Contraste`), aplica o pipeline escolhido (**CLAHE + Ajuste de Contraste**) salvando em `data/processed/mod_spectograms/` e gera o arquivo `data/annotations/modifications.csv`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.image_enhancer import (
    apply_histogram_equalization,
    apply_clahe,
    apply_brightness,
    apply_contrast,
    enhance_spectrogram,
)

%matplotlib inline
import matplotlib.pyplot as plt

ORIG_DIR    = PROJECT_ROOT / 'data' / 'processed' / 'spectrograms'
MOD_DIR     = PROJECT_ROOT / 'data' / 'processed' / 'mod_spectograms'
CSV_PATH    = PROJECT_ROOT / 'data' / 'annotations' / 'modifications.csv'
SAMPLE_PATH = ORIG_DIR / 'PN00' / 'PN00-1_seizure01_ictal.png'

sample_img = np.array(Image.open(SAMPLE_PATH).convert('RGB'), dtype=np.uint8)

In [ ]:
# Comparacao visual das 4 tecnicas e do Pipeline Final
comparacoes = [
    ('Original', sample_img),
    ('1. Equalizacao Global (Descartada)', apply_histogram_equalization(sample_img)),
    ('2. CLAHE (Mantida)', apply_clahe(sample_img)),
    ('3. Ajuste de Brilho (Descartada)', apply_brightness(sample_img)),
    ('4. Ajuste de Contraste (Mantida)', apply_contrast(sample_img)),
    ('Pipeline Final (CLAHE + Contraste)', enhance_spectrogram(sample_img)),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, (titulo, img) in zip(axes.flat, comparacoes):
    ax.imshow(img, aspect='auto', origin='lower')
    ax.set_title(f'{titulo}\n(media={img.mean():.2f}, std={img.std():.2f})')
    ax.set_xlabel('Tempo (frames STFT)')
    ax.set_ylabel('Frequencia (bins)')

plt.suptitle(f'Comparacao das Tecnicas de Melhoria Visual — {SAMPLE_PATH.name}', fontweight='bold')
plt.tight_layout()
plt.show()

### Justificativa das escolhas
- **Equalização Global (Descartada):** Como a imagem possui ~87% de pixels zero, a CDF salta abruptamente e amplifica ruído de baixa energia.
- **Ajuste de Brilho (Descartada):** Desloca o fundo preto (`0`) para cinza sem aumentar a separação real entre as faixas espectrais.
- **CLAHE + Ajuste de Contraste (Mantidas):** Realçam localmente as descargas epilépticas e expandem a faixa dinâmica preservando integralmente o fundo zero.

In [ ]:
# Gera as imagens modificadas em data/processed/mod_spectograms e o arquivo modifications.csv
def parse_class(stem: str) -> str:
    for cls in ['pre_ictal', 'pos_ictal', 'normal', 'ictal']:
        if stem.endswith('_' + cls):
            return cls
    return 'unknown'

orig_files = sorted(ORIG_DIR.rglob('*.png'))
records = []

for idx, orig_path in enumerate(tqdm(orig_files, desc='Processando imagens'), start=1):
    rel_path = orig_path.relative_to(ORIG_DIR)
    mod_path = MOD_DIR / rel_path
    mod_path.parent.mkdir(parents=True, exist_ok=True)

    orig_arr = np.array(Image.open(orig_path).convert('RGB'), dtype=np.uint8)
    mod_arr  = enhance_spectrogram(orig_arr)
    Image.fromarray(mod_arr).save(mod_path)

    o_mean, m_mean = float(orig_arr.mean()), float(mod_arr.mean())
    o_std, m_std   = float(orig_arr.std()), float(mod_arr.std())

    records.append({
        'id': idx,
        'patient_id': rel_path.parts[0],
        'class': parse_class(orig_path.stem),
        'filename': orig_path.name,
        'original_path': str(orig_path.relative_to(PROJECT_ROOT)),
        'modified_path': str(mod_path.relative_to(PROJECT_ROOT)),
        'techniques_applied': 'CLAHE + Ajuste de Contraste',
        'orig_mean': round(o_mean, 4),
        'mod_mean': round(m_mean, 4),
        'orig_std': round(o_std, 4),
        'mod_std': round(m_std, 4),
        'modification_description': (
            f'CLAHE (clip=2.5, grid=8x8) + Contraste (alpha=1.35, gamma=0.85); '
            f'media {o_mean:.2f}->{m_mean:.2f}, std {o_std:.2f}->{m_std:.2f}'
        ),
    })

df_mod = pd.DataFrame(records)
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
df_mod.to_csv(CSV_PATH, index=False)
print(f'Imagens salvas em {MOD_DIR.relative_to(PROJECT_ROOT)} e CSV salvo em {CSV_PATH.relative_to(PROJECT_ROOT)} ({len(df_mod)} registros)')
display(df_mod.head())